[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/01_SupervisedLearning.ipynb#copy=true)


# Supervised Learning: Regression, Evaluation, and Generalization

## Learning objectives

By the end of this lesson, you should be able to:

- Distinguish supervised learning from unsupervised learning.
- Fit and interpret simple and multiple linear regression models.
- Separate training and test data, and use mean squared error to evaluate predictions.
- Explain why a model that performs well on training data may not generalize.

**Student Learning Outcome (SLO 2):**
> Build and assess a supervised learning model using an appropriate data split, error metric, and written interpretation.


## Before we begin

Suppose you want to predict a student's final course score from their prior coursework, time spent studying, and attendance. Which quantity is the response $y$? Which quantities are features $X$? What information would be missing if you only reported the model's training accuracy?

In **supervised learning**, each observation has features $X$ and a known target $y$. We learn a rule $\hat f(X)$ from labeled examples, then assess how well it predicts labels we did not use to fit it. Regression predicts a numerical target; classification predicts a category.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score

rng = np.random.default_rng(232)
x = rng.uniform(0, 10, 80)
y = 4 + 2.5 * x + rng.normal(0, 3, size=x.size)
plt.scatter(x, y, alpha=0.75)
plt.xlabel("feature x")
plt.ylabel("response y")
plt.show()


## Part 1 — Simple linear regression

For one feature, linear regression uses $\hat y = \hat\beta_0 + \hat\beta_1x$. It selects the coefficients that minimize the residual sum of squares,

$$RSS = \sum_{i=1}^n (y_i - \hat y_i)^2.$$

The gradient-descent idea from the earlier Module 2 material is one way to minimize this loss. `LinearRegression` performs the least-squares fitting for us, so we can focus on modeling and evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1, 1), y, test_size=0.25, random_state=232
)
model = LinearRegression().fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"intercept: {model.intercept_:.2f}")
print(f"slope: {model.coef_[0]:.2f}")
print(f"test MSE: {mean_squared_error(y_test, predictions):.2f}")

grid = np.linspace(0, 10, 100).reshape(-1, 1)
plt.scatter(X_train, y_train, label="training data")
plt.scatter(X_test, y_test, label="test data")
plt.plot(grid, model.predict(grid), color="black", label="fitted line")
plt.legend(); plt.show()


### ✏️ Written response 1

In 3–5 sentences, interpret the slope. Then explain why the test MSE—not just the training MSE—is relevant when deciding whether this model is useful.

> **YOUR ANSWER:**


## Part 2 — Multiple linear regression and model selection

With $p$ features, the model becomes $\hat y = \hat\beta_0 + \hat\beta_1x_1 + \cdots + \hat\beta_px_p$. The diabetes data contain ten standardized predictors and a quantitative disease-progression response. This is the same dataset used in the prior principal-components-regression lesson.


In [ ]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=232)

multiple_model = LinearRegression().fit(X_train, y_train)
test_pred = multiple_model.predict(X_test)
print(f"test MSE: {mean_squared_error(y_test, test_pred):.1f}")
print(f"test R²:  {r2_score(y_test, test_pred):.3f}")

# Five-fold cross-validation estimates out-of-sample error using several splits.
cv_mse = -cross_val_score(LinearRegression(), X, y, cv=5, scoring="neg_mean_squared_error")
print(f"5-fold CV MSE: {cv_mse.mean():.1f} ± {cv_mse.std():.1f}")


### Practice

1. Fit a model using only the first two predictors. Compare its test MSE with the full model.
2. Why can adding predictors lower training error while increasing test error?
3. What would you check before using this model to make a consequential decision about a person?

> **YOUR ANSWER:**


## Key ideas

- A held-out test set estimates performance on new data.
- Cross-validation is helpful when one train/test split would be unstable.
- Coefficients describe associations in the data; they do not by themselves establish causation.
- Next, we apply the same feature-matrix idea to text.


## Part 3 — Residuals and the loss function

A **residual** is $e_i = y_i - \hat y_i$: the signed vertical distance from an observation to a prediction. Squaring residuals makes positive and negative errors comparable and penalizes large misses. Mean squared error is

$$MSE = \frac{1}{n}\sum_{i=1}^n (y_i - \hat y_i)^2.$$

Least squares finds coefficients that minimize this quantity. Gradient descent reaches the same goal iteratively by taking small steps opposite the gradient of the loss.


In [ ]:
training_predictions = model.predict(X_train)
residuals = y_train - training_predictions
print(f"training MSE: {mean_squared_error(y_train, training_predictions):.2f}")
plt.axhline(0, color="black", linewidth=1)
plt.scatter(training_predictions, residuals)
plt.xlabel("predicted response"); plt.ylabel("residual (actual − predicted)")
plt.show()


### Check your understanding

1. What would a residual plot look like if the model underpredicted large values?
2. Why does MSE react strongly to outliers?
3. Does a zero training residual prove perfect prediction on new data?

> **YOUR ANSWER:**


## Part 4 — Regression versus classification

Regression predicts a numerical value, such as temperature or sale price. Classification predicts a label, such as spam/not spam. For a binary classifier, changing the probability threshold trades false positives against false negatives; accuracy is therefore not always a sufficient success measure.


### Exit ticket

Describe one supervised-learning problem from your field. Name its features, target, problem type, and one especially costly error.

> **YOUR ANSWER:**


## Part 5 — One gradient-descent step

For simple regression, gradient descent updates each parameter using a learning rate $\eta$:

$$\beta_j \leftarrow \beta_j - \eta\frac{\partial MSE}{\partial \beta_j}.$$

A small $\eta$ makes slow progress; a large $\eta$ can overshoot the minimum. In practice, numerical libraries solve least squares efficiently, but the update rule explains how many modern learning algorithms improve a model.


### Small-group task

Sketch a curve of loss versus iteration for a learning rate that is too small, reasonable, and too large. What evidence would tell you to stop training?

> **YOUR ANSWER:**


## Model-assessment checklist

Before trusting a supervised model, ask: Is the target measured reliably? Is the test set genuinely future or unseen data? Are key groups represented? Is the metric aligned with the decision? Could a feature leak information that would be unavailable at prediction time?

Choose one checklist question that matters for the diabetes example and explain why.

> **YOUR ANSWER:**


## Lesson summary

Supervised learning connects labeled features to a target. Fitting is only the beginning: residuals, train/test splits, cross-validation, and context determine whether a model is credible.
